In [ ]:
# Colab setup
import sys
if "google.colab" in sys.modules:
    %pip install -q pyedautils ephem statsmodels plotly kaleido ipywidgets

import plotly.io as pio
pio.renderers.default = "notebook_connected"

# Comfort Compass

```{admonition} Goal
:class: tip
Show the distribution of days across **nine comfort states** (cold / ok / warm × dry / ok / humid) with **three severity stages**, as an *area-true* polar "compass": at a glance which way — and how strongly — the indoor climate drifts out of the comfort band.
```

```{admonition} Data Basis
:class: note
We use the bundled dataset `room_climate_sample.csv` shipped with **pyedautils** —
anonymised hourly temperature and humidity from a real sensor in a cool,
humidity-controlled room. It is loaded directly from the installed package
(`importlib.resources`), so no download is needed.
```

## Data Preparation

In [ ]:
import pandas as pd
from importlib import resources

data_path = resources.files("pyedautils") / "data" / "room_climate_sample.csv"
df = pd.read_csv(data_path)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.set_index("timestamp")   # DatetimeIndex -> the compass counts days

df.head()

## Visualization

In [ ]:
from pyedautils.comfort import comfort_compass_distribution
from pyedautils.plots import plot_comfort_compass

# Cool, humidity-controlled room -> target band 8-18 deg C / 40-70 %rH.
dist = comfort_compass_distribution(df, temp_band=(8, 18), hum_band=(40, 70))
fig = plot_comfort_compass(dist, title="Indoor comfort compass")
fig.show()

::::{dropdown} Customization
The green centre is the share of days inside the band; each spoke's **area** is the
share of days in that direction, split into *mild / moderate / severe* stages
(saturation light → dark). Tune the bands and severity limits, or also judge humidity
by its **absolute** limit with `hum_abs_band` (a warm point above the absolute-humidity
cap then reads as "warm + humid" even if its relative humidity is still in range).

```python
dist = comfort_compass_distribution(
    df,
    temp_band=(8, 18), hum_band=(40, 70),
    hum_abs_band=(0.0, 0.0115),       # also cap absolute humidity [kg/kg]
    temp_stage_limits=(1.0, 2.5),     # K over the band: mild / moderate
    hum_stage_limits=(5.0, 10.0),     # %-points over the band
)
fig = plot_comfort_compass(dist, title="Indoor comfort compass")
```
::::